# Capitolo 8: Pipeline Completa — FlowStitch in Azione

## Obiettivo
Dimostrare il funzionamento end-to-end della pipeline FlowStitch su dati reali,
utilizzando il codice del pacchetto `flowstitch/`.

## Fasi della Pipeline
1. Caricamento dati dal dataset (tensori pre-estratti da FLUX.1-schnell)
2. Compilazione maschera (attenzione, spettrale, ibrida, TDA)
3. Latent Stitching con KTS + EMA
4. Valutazione qualitativa e quantitativa

## Prerequisiti
- Capitoli 1-7: l'intero percorso di ricerca


In [ ]:
import os, sys, torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from flowstitch.core.config import FlowStitchConfig
from flowstitch.core.tokenizer_utils import find_token_indices
from flowstitch.extraction import (
    extract_attention_mask, otsu_threshold,
    compute_fiedler_mask, compute_chebyshev_threshold,
    hybrid_semantic_decomposition, extract_tda_mask
)
from flowstitch.stitching.kts import apply_kts, compute_damping_factor
from flowstitch.stitching.ema_smoothing import AttentionEMA
from flowstitch.evaluation.metrics import dice_coefficient, iou_score

print("Tutti i moduli FlowStitch importati correttamente.")


## 8.1 Caricamento del Dataset Reale

I tensori sono stati pre-estratti da FLUX.1-schnell con 4 step di inferenza.
Ogni campione contiene:
- `attention_maps.pt`: mappe di cross-attention $[1, 24, 4096, 512]$
- `v0_velocity.pt`: campo vettoriale $v_0$ a $t=0$ $[1, 4096, 64]$
- `x0_noise.pt`: rumore iniziale $x_0$ $[1, 4096, 64]$


In [ ]:
# Caricamento di una scena multi-oggetto
db_path = "../data/dataset_v1/a_blue_cube_and_a_red_sphere"

attn_maps = torch.load(os.path.join(db_path, "attention_maps.pt"), map_location="cpu", weights_only=True)
v0 = torch.load(os.path.join(db_path, "v0_velocity.pt"), map_location="cpu", weights_only=True)
x0 = torch.load(os.path.join(db_path, "x0_noise.pt"), map_location="cpu", weights_only=True)

print(f"Attention maps: {attn_maps.shape}")
print(f"Velocity field v0: {v0.shape}")
print(f"Initial noise x0: {x0.shape}")


## 8.2 Confronto dei Metodi di Estrazione Maschera

Applichiamo tutti i metodi implementati in `flowstitch.extraction` allo stesso campione.


In [ ]:
from transformers import T5Tokenizer

prompt = "a blue cube and a red sphere"
target_word = "cube"

try:
    tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl", legacy=True)
    token_indices = find_token_indices(tokenizer, prompt, target_word)
    print(f"Token indices per '{target_word}': {token_indices}")
except Exception as e:
    print(f"Tokenizer non disponibile, uso indici manuali: {e}")
    token_indices = [3]

layer_10_attn = attn_maps  # già layer 10 nel dataset

# 1. Maschera di Attenzione Basica
attn_mask = extract_attention_mask(layer_10_attn, token_indices)
print(f"Attention mask: {attn_mask.shape}, range [{attn_mask.min():.3f}, {attn_mask.max():.3f}]")

# 2. Otsu Thresholding
otsu_mask = otsu_threshold(attn_mask)
print(f"Otsu mask: pixel attivi = {otsu_mask.sum().item():.0f}/{otsu_mask.numel()}")

# 3. Chebyshev Energy Gating
energy_map = torch.norm(v0, p=2, dim=-1, keepdim=True)
chebyshev_mask = compute_chebyshev_threshold(energy_map, k=1.0)
print(f"Chebyshev mask: pixel attivi = {chebyshev_mask.sum().item():.0f}")

# 4. Spectral (Fiedler)
fiedler_mask = compute_fiedler_mask(v0, target_resolution=32)
print(f"Fiedler mask: pixel attivi = {fiedler_mask.sum().item():.0f}")

# 5. Hybrid (Spectral + Attention)
hybrid_mask = hybrid_semantic_decomposition(v0, attn_mask, target_resolution=32)
print(f"Hybrid mask: pixel attivi = {hybrid_mask.sum().item():.0f}")

# 6. TDA
tda_mask = extract_tda_mask(v0, attn_mask)
print(f"TDA mask: pixel attivi = {tda_mask.sum().item():.0f}")


In [ ]:
# Visualizzazione comparativa
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

masks = [
    ("Attenzione (raw)", attn_mask),
    ("Otsu Threshold", otsu_mask),
    ("Chebyshev (μ+σ)", chebyshev_mask.float()),
    ("Fiedler (Spettrale)", fiedler_mask.float()),
    ("Hybrid (Spettrale+Attn)", hybrid_mask.float()),
    ("TDA (Persistent H₀)", tda_mask.float()),
]

for ax, (title, mask) in zip(axes.flat, masks):
    img = mask.squeeze().view(64, 64).detach().numpy()
    ax.imshow(img, cmap='viridis')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.suptitle("Confronto Metodi di Estrazione — 'cube' in 'a blue cube and a red sphere'", 
             fontsize=16, fontweight='bold')
plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/mask_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 8.3 Kinetic Trajectory Shaping (KTS)

Il meccanismo di damping che attenua la perturbazione negli step finali dell'ODE:
$$D(t) = \exp(-\gamma \cdot \max(0, t - t_{\text{cutoff}}))$$

$$v_{\text{stitch}} = v_{\text{ambient}} + D(t) \cdot \lambda \cdot [M \odot (v_{\text{target}} - v_{\text{ambient}})]$$


In [ ]:
import numpy as np

# Curva di damping KTS
t_values = np.linspace(0, 1, 100)
damping = [compute_damping_factor(t, t_cutoff=0.8, gamma=5.0) for t in t_values]

plt.figure(figsize=(10, 5))
plt.plot(t_values, damping, 'b-', linewidth=2)
plt.axvline(x=0.8, color='r', linestyle='--', alpha=0.5, label='$t_{cutoff} = 0.8$')
plt.fill_between(t_values, damping, alpha=0.1)
plt.xlabel('Tempo normalizzato $t$', fontsize=12)
plt.ylabel('Fattore di damping $D(t)$', fontsize=12)
plt.title('KTS: Attenuazione Esponenziale della Perturbazione', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.savefig('../outputs/kts_damping.png', dpi=150, bbox_inches='tight')
plt.show()

# Applicazione KTS su un singolo step
v_ambient = torch.randn_like(v0) * 0.1
mask_kts = hybrid_mask.float()

v_stitched = apply_kts(
    v_ambient=v_ambient, v_target=v0, mask=mask_kts,
    t_norm=0.0, lambda_val=1.0, t_cutoff=0.8, gamma=5.0
)
print(f"v_stitched shape: {v_stitched.shape}")
print(f"Perturbazione media: {(v_stitched - v_ambient).abs().mean().item():.6f}")


## 8.4 EMA Smoothing Temporale

$$\bar{A}_t = \gamma \cdot A_t + (1 - \gamma) \cdot \bar{A}_{t-1}$$


In [ ]:
# Simulazione EMA su 10 step
smoother = AttentionEMA(decay=0.3)

step_maps = []
for step in range(10):
    noisy_map = attn_mask + torch.randn_like(attn_mask) * 0.1 * (1 - step/10)
    smoothed = smoother.update(noisy_map)
    step_maps.append(smoothed.squeeze().view(64, 64).detach().numpy())

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, (ax, smap) in enumerate(zip(axes.flat, step_maps)):
    ax.imshow(smap, cmap='viridis')
    ax.set_title(f'Step {i}', fontsize=11)
    ax.axis('off')

plt.suptitle('EMA Smoothing — Convergenza Temporale', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/ema_smoothing.png', dpi=150, bbox_inches='tight')
plt.show()


## 8.5 Architettura della Pipeline

```
FLUX.1 Inference → FluxDataCapturer → Dataset (x0, v0, attn_maps)
                                          ↓
                              Mask Compilation
                         (Attn / Spectral / Hybrid / TDA)
                                          ↓
                              Latent Stitching
                         (KTS + SemanticGrafting + EMA)
                                          ↓
                              VAE Decode → Immagine Finale
```
